In [2]:
!pip install tensorflow_recommenders
!pip install tensorflow_datasets
!pip install tentensorflow

ERROR: Could not find a version that satisfies the requirement tentensorflow (from versions: none)
ERROR: No matching distribution found for tentensorflow


In [3]:
import os
import numpy
from typing import Dict, Text
import tensorflow.keras as keras
import tensorflow as tf
from abc import ABC
import tensorflow_recommenders as tfrs
import tensorflow_datasets as tfds

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

In [ ]:

ratings = tfds.load('movielens/100k-ratings', split='train')
movies = tfds.load('movielens/100k-movies', split='train')

ratings = ratings.map(lambda x: {'user_id': x['user_id'], 'movie_title': x['movie_title'], 'user_rating': x['user_rating']})
movies = movies.map(lambda x: x['movie_title'])

In [ ]:
tf.random.set_seed(69)   # Nice

shuffled = ratings.shuffle(100000, seed=10, reshuffle_each_iteration=False)

Train = shuffled.take(80000)
Test = shuffled.skip(80000).take(20000)

movie_titles = movies.batch(1000)
user_ids = ratings.batch(1000).map(lambda x: x['user_id'])

unique_movie_titles = numpy.unique(numpy.concatenate(list(movies.batch(1000))))
unique_user_ids = numpy.unique(numpy.concatenate(list(user_ids.batch(1000))))

In [ ]:
class MovieLensModel(tfrs.Model, ABC):
    def __init__(self, rating_weight: float, retrieval_weight: float):
        super().__init__()

        _embedding_dim = 32


        self._movie_model = keras.Sequential([
            keras.layers.StringLookup(vocabulary=unique_movie_titles, mask_token=None),
            keras.layers.Embedding(len(unique_movie_titles) + 1, _embedding_dim)
        ])


        self._user_model = keras.Sequential([
            keras.layers.StringLookup(vocabulary=unique_user_ids, mask_token=None),
            keras.layers.Embedding(len(unique_user_ids) + 1, _embedding_dim)
        ])


        self._rating_model = keras.Sequential([
            keras.layers.Dense(256, activation='relu'),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dense(1),
        ])

        self._rating_task = tfrs.tasks.Ranking(
            loss=keras.losses.MeanSquaredError(),
            metrics=[keras.metrics.RootMeanSquaredError()],
            name='mse',
        )

        self._retrieval_task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=movies.batch(128).map(self._movie_model),
                name='top_k',
            )
        )

        self._rating_weight = rating_weight
        self._retrieval_weight = retrieval_weight

    def call(self, features: Dict[Text, tf.Tensor]):
        user_embeddings = self._user_model(features['user_id'])
        movie_embeddings = self._movie_model(features['movie_title'])

        return (
            user_embeddings,
            movie_embeddings,
            self._rating_model(
                tf.concat([user_embeddings, movie_embeddings], axis=1)
            ),
        )

    def compute_loss(self, features: Dict[Text, tf.Tensor], training: bool = False):
        ratings = features.pop('user_rating')
        user_embeddings, movie_embeddings, rating_predictions = self(features)

        ratings_loss = self._rating_task(labels=ratings, predictions=rating_predictions)
        retrieval_loss = self._retrieval_task(user_embeddings, movie_embeddings)

        return (self._rating_weight * ratings_loss + self._retrieval_weight * retrieval_loss)

    @property
    def user_model(self):
        return self._user_model

    @property
    def movie_model(self):
        return self._movie_model

    @property
    def task(self):
        return self._task

In [ ]:

model = MovieLensModel(rating_weight=1.0, retrieval_weight=1.0)
model.compile(optimizer=keras.optimizers.Adagrad(0.1))

cached_train = Train.shuffle(100000).batch(8192).cache()
cached_test = Test.batch(4096).cache()

model.fit(cached_train, epochs=3)

metrics = model.evaluate(cached_test, return_dict=True)
print(metrics)

trained_movie_embedding, trained_user_embeddings, predicted_ratings = model({
    'user_id': numpy.array(['13']),
    'movie_title': numpy.array(['Dance with the Wolves (1990)'])
})

print(f'Predicted Rating : {predicted_ratings}')


model.compile(optimizer=keras.optimizers.Adagrad(0.1, name="ada_grad"))
model.fit(Train.batch(8192), epochs=3)


brute_force = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
brute_force.index_from_dataset(movies.batch(128).map(lambda title: (title, model.movie_model(title))))

_, titles = brute_force(numpy.array(['10']), k=3)
print(f'Titles: {titles[0]}')